# ComfyUI GPU endpoint
Select a GPU runtime. This notebook installs open-source ComfyUI on the hosted Colab runtime and exposes it through a temporary Cloudflare tunnel. Treat the URL as a secret; the runtime and URL are ephemeral.

In [ ]:
!nvidia-smi
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
import subprocess, time, re
server=subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188'],cwd='/content/ComfyUI')
time.sleep(8)
tunnel=subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8188','--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
for line in tunnel.stdout:
 print(line,end='')
 m=re.search(r'https://[-a-z0-9]+\.trycloudflare.com',line)
 if m:
  print('\nCOMFY_URL='+m.group(0)); break

Place checkpoints in `/content/ComfyUI/models/checkpoints`, LoRAs in `/content/ComfyUI/models/loras`, and API-format workflows in the Termux project. Large files disappear when the runtime ends unless mounted to Drive or downloaded again from a trusted registry.